In [1]:
import os
import numpy as np
from data_utils import load_tokens

In [2]:
BASE_DATA_PATH = "ft_datasets"
ALPACA_DATA_PATH = os.path.join(BASE_DATA_PATH, "alpaca") 

In [3]:
train_tokens = load_tokens(os.path.join(ALPACA_DATA_PATH, "splits/train.bin"))
val_tokens = load_tokens(os.path.join(ALPACA_DATA_PATH, "splits/val.bin"))
test_tokens = load_tokens(os.path.join(ALPACA_DATA_PATH, "splits/test.bin"))

In [4]:
train_tokens[:10]

tensor([50256, 21106,   318,   281, 12064,   326,  8477,   257,  4876,    11])

In [5]:
import tiktoken
enc = tiktoken.encoding_for_model("gpt-2")

def decode_tokens(tokens):
    return "".join(enc.decode(tokens))


In [6]:
# so we want to create a DataLoader
# lets not care about masking for now

class AlpacaFineTuneDataLoader:
    def __init__(self, B, T, num_processes=1, process_rank=0, device="cpu", split="train"):
        """
        B: batch size
        T: sequence length
        num_processes: number of processes used in distributed training
        process_rank: rank of the current process
        device: device to use
        split: "train", "val", or "test"
        """

        self.B = B
        self.T = T
        self.process_rank = process_rank
        self.num_processes = num_processes
        self.device = device
        assert split in {"train", "val", "test"}
        data_folder = os.path.join(ALPACA_DATA_PATH, "splits")
        self.tokens = load_tokens(os.path.join(data_folder, f"{split}.bin"))
        
        master_process = process_rank == 0
        if master_process:
            print(f"Loaded {len(self.tokens)} tokens for {split} split")
        
        # imagine we concatenate batches used across all processes for one big forward pass in distributed training
        self.current_idx = self.B * self.T * process_rank
    
    def reset(self):
        self.current_idx = self.B * self.T * self.process_rank
    
    def next_batch(self):
        """
        Returns a batch of size B x T
        """
        B, T = self.B, self.T

        buf = self.tokens[self.current_idx:self.current_idx + (B * T) + 1]
        x = buf[:-1].view(B, T) # inputs
        y = buf[1:].view(B, T) # targets

        self.current_idx += B*T*self.num_processes
        if self.current_idx + (B*T*self.num_processes) + 1 >= len(self.tokens):
            self.reset()
        
        return x.to(self.device), y.to(self.device)

In [7]:
# lets test the DataLoader
B = 2
T = 10
dataloader = AlpacaFineTuneDataLoader(B, T, split="val")
x, y = dataloader.next_batch()
print(x.shape, y.shape)

Loaded 1043320 tokens for val split
torch.Size([2, 10]) torch.Size([2, 10])
